# Dependencies

In [8]:
import json
import math
import os
import re
import time
from collections import Counter
from dataclasses import dataclass
import pandas as pd
import ast

import numpy as np
import requests
import torch
import torch.nn as nn
from torch.nn import functional as F


MARKER = "Story:"

PAD = "[PAD]"
EOT = "<|endoftext|>"
VOCAB_SIZE = 10000
SPECIALS = ("[UNK]", PAD, EOT)  # reserved tokens; take ids 0,1,2
SPECIALS_RE = re.compile("(" + "|".join(re.escape(s) for s in SPECIALS) + ")")

device = "cpu"

if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(0)

BASE_PATH = "./"


block_size = 256
n_layer = 6
n_head = 6
n_embd = 384


In [9]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Projects/Tutorials/LM/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data

In [10]:
RAW = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
INS = "https://huggingface.co/datasets/roneneldan/TinyStoriesInstruct/resolve/main"
CAP = 80 * 1024 * 1024  # ~80MB per big train file

os.makedirs(os.path.join(BASE_PATH, "data"), exist_ok=True)

JOBS = [
    (f"{RAW}/TinyStoriesV2-GPT4-train.txt", "pretrain_train.txt", CAP),
    (f"{RAW}/TinyStoriesV2-GPT4-valid.txt", "pretrain_valid.txt", None),
    (f"{INS}/TinyStories-Instruct-train.txt", "instruct_train.txt", CAP),
    (f"{INS}/TinyStories-Instruct-valid.txt", "instruct_valid.txt", None),
]


def download(url, dest, cap):
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    data = bytearray()
    for chunk in r.iter_content(1 << 20):
        data += chunk
        if cap and len(data) >= cap:
            break
    r.close()
    text = data.decode("utf-8", errors="ignore")
    if cap:  # cut back to the last whole story
        text = text[: text.rfind(EOT) + len(EOT)]
    with open(dest, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"{os.path.basename(dest)}: {len(text)/1e6:.1f} MB, ~{text.count(EOT)} stories")


for url, name, cap in JOBS:
  dest = os.path.join(BASE_PATH, "data", name)
  if os.path.exists(dest) and os.path.getsize(dest):
      print(f"{dest}: already present, skipping")
      continue
  download(url, dest, cap)




/content/drive/MyDrive/Projects/Tutorials/LM/data/pretrain_train.txt: already present, skipping
/content/drive/MyDrive/Projects/Tutorials/LM/data/pretrain_valid.txt: already present, skipping
/content/drive/MyDrive/Projects/Tutorials/LM/data/instruct_train.txt: already present, skipping
/content/drive/MyDrive/Projects/Tutorials/LM/data/instruct_valid.txt: already present, skipping


# Tokenizer

In [11]:
"""A word-level tokenizer, implemented from scratch (no external tokenizer lib).

Pipeline, mirroring what production tokenizers do, just simpler:

  1. pre-tokenize: split raw text into word / punctuation pieces with one regex
  2. build vocab: keep the most frequent pieces (above a min count), up to a cap
  3. encode: peel off special tokens first (so they map to their own id instead
     of being shredded by the regex), pre-tokenize the rest, map pieces -> ids,
     and send any piece not in the vocab to [UNK]
  4. decode: map ids -> pieces, drop special tokens, join with spaces

Special tokens occupy the first ids: [UNK]=0, [PAD]=1, <|endoftext|>=2.
[PAD] (id 1) never appears in data; it's reserved so we can pad variable-length
sequences into a rectangular batch (the loss ignores those positions).
<|endoftext|> marks a document boundary / "stop here".

Why WORD-LEVEL is a teaching choice with real costs — this is precisely why
subword tokenizers exist:
  * OOV / information loss: any word not in the vocab collapses to a single
    [UNK] and is unrecoverable, so decode(encode(text)) is lossy for rare words.
  * Vocab bloat: "Dog", "dog", "dogs", "dogged" are unrelated ids with no shared
    structure, so the table fills with case/morphology duplicates (~12% of this
    10k vocab) yet still can't spell a word it never saw.
  * BPE/subword fixes both: it learns character-merge rules, so novel words
    decompose into known sub-pieces and morphology is shared. That's what
    GPT-2 / tiktoken / sentencepiece use; word-level is the simplest thing that
    works and the clearest way to FEEL why subwords were invented.

Vocab is saved as plain JSON so it's inspectable.
"""

# words (letters/digits/underscore) OR a run of punctuation. Matches the common
# "Whitespace" pre-tokenizer behavior: "Hello!" -> ["Hello", "!"], keeping case.


PRETOK = re.compile(r"\w+|[^\w\s]+")  # \w+ = a word; [^\w\s]+ = a run of punctuation chars
def pretokenize(text):               # split raw text into word/punctuation pieces
    return PRETOK.findall(text)     # every regex match, in order, e.g. ["Hello","!"]

class WordTokenizer:
    def __init__(self, vocab):   # build a tokenizer from a vocab list
        self._special_set = set(SPECIALS)      # set for O(1) "is this a special?" checks
        self.id_to_token = list(vocab)              # id -> token string (index = token id)
        self.token_to_id_map = {t: i for i, t in enumerate(self.id_to_token)}  # token string -> id
        # [UNK] is the OOV fallback; default to id 0 if a custom specials list
        # omits it, rather than crashing at construction.
        self.unk_id = self.token_to_id_map.get("[UNK]", 0)  # id emitted for unknown words
        # A matcher for whole special tokens, so encode() can peel them off before
        # the regex pre-tokenizer would shred e.g. "<|endoftext|>" into "<|" etc.

    # ---- training ----------------------------------------------------------
    @classmethod
    def train(cls, paths, vocab_size=10000, min_frequency=2):
        counts = Counter()                          # token string -> number of occurrences
        for p in paths:                             # for each corpus file path
            with open(p, "r", encoding="utf-8") as f:   # open it for reading (utf-8)
                for line in f:                      # stream line by line (keeps memory flat)
                    # don't let the doc separator pollute word statistics
                    counts.update(pretokenize(line.replace(EOT, " ")))  # tally this line's pieces
        for s in SPECIALS:           # never let a special compete on frequency
            counts.pop(s, None)                     # remove it from the counts if it appeared
        # most frequent first; ties broken alphabetically for determinism
        ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))  # sort by (-count, token)
        vocab = list(SPECIALS)                      # specials occupy the first ids (0,1,2)
        for tok, c in ranked:                       # walk pieces from most to least frequent
            if len(vocab) >= vocab_size:            # stop once we hit the size cap
                break
            if c < min_frequency:                   # stop once below the frequency floor
                break                               # (ranked is sorted, so everything after is too)
            vocab.append(tok)                       # otherwise keep this token
        return cls(vocab)        # construct the tokenizer from the final vocab

    # ---- encode / decode ---------------------------------------------------
    def encode(self, text):                         # text -> list[int] of token ids
        m, unk = self.token_to_id_map, self.unk_id  # local aliases (slightly faster in the loop)
        # Split out any special tokens first so they map to their own id; the
        # text between them goes through the normal word/punctuation pre-tokenizer.
        parts = SPECIALS_RE.split(text)
        ids = []                                    # accumulator for the output ids
        for part in parts:                          # each chunk is either a special token or plain text
            if not part:                            # re.split can yield empty strings
                continue                            # skip them
            if part in self._special_set:           # if this whole chunk is a special token
                ids.append(m[part])                 # emit its single id
            else:                                   # otherwise it's ordinary text
                ids.extend(m.get(t, unk) for t in pretokenize(part))  # map each piece; OOV -> unk
        return ids                                  # the encoded id sequence

    def encode_batch(self, texts):                  # encode a list of strings
        return [self.encode(t) for t in texts]      # encode() applied to each, preserving order

    def decode(self, ids, skip_special=True):       # list[int] -> a text string
        n = len(self.id_to_token)                       # vocab size, used for range checks
        out = []                                    # collected token strings
        for i in ids:                               # for each id in the sequence
            if i < 0 or i >= n:                     # defensively ignore out-of-range ids
                continue
            t = self.id_to_token[i]                             # the token string for this id
            if skip_special and t in self._special_set:  # by default drop [UNK]/[PAD]/<|endoftext|>
                continue
            out.append(t)                           # keep real tokens
        return " ".join(out)                        # join with spaces (word-level detokenization)

    # ---- lookups / io ------------------------------------------------------
    def token_to_id(self, token):                   # look up the id of a single token
        return self.token_to_id_map.get(token)      # returns None if the token isn't in the vocab

    def size(self):                       # number of tokens in the vocabulary
        return len(self.id_to_token)

    def save(self, path):                           # write the vocab + specials to JSON
        with open(path, "w", encoding="utf-8") as f:    # open the destination file for writing
            json.dump({"vocab": self.id_to_token},  # everything needed to rebuild
                      f, ensure_ascii=False)        # keep unicode readable (don't \u-escape it)

    @classmethod
    def from_file(cls, path):                       # load a tokenizer previously written by save()
        with open(path, "r", encoding="utf-8") as f:    # open the JSON file
            d = json.load(f)                        # parse it into a dict
        return cls(d["vocab"])  # rebuild the tokenizer


In [12]:
FILES = [os.path.join(BASE_PATH,"data", "pretrain_train.txt"), os.path.join(BASE_PATH, "data", "instruct_train.txt")]  # corpora to learn the vocab from


# Count every word/punctuation piece across FILES, drop pieces seen < 2 times,
# keep the most frequent up to VOCAB_SIZE; returns a ready WordTokenizer.
tok = WordTokenizer.train(FILES, vocab_size=VOCAB_SIZE, min_frequency=2)
tok.save(os.path.join(BASE_PATH, "tokenizer.json"))


In [13]:
pd.DataFrame({
    'ID': range(len(tok.id_to_token)),
    'Token': tok.id_to_token
})

,ID,Token
0,0,[UNK]
1,1,[PAD]
2,2,<|endoftext|>
3,3,.
4,4,","
...,...,...
9995,9995,Grumbly
9996,9996,Guard
9997,9997,Hanger
9998,9998,Heli


In [14]:
sample = 'Once upon a time, a cat said, "Hello!"'   # a string to demo encode/decode on
ids = tok.encode(sample)               # encode: text -> list of integer token ids
print("sample:", sample)              # show the original string
print("encoded:", ids)                # show the token ids
print("decoded:", tok.decode(ids))    # decode: list of token ids -> text

sample: Once upon a time, a cat said, "Hello!"
encoded: [60, 61, 7, 47, 4, 7, 100, 19, 4, 11, 449, 58]
decoded: Once upon a time , a cat said , " Hello !"


# GPT Model

In [15]:
"""A small GPT, coded from scratch. nanoGPT-style decoder-only transformer.

Single file, no external model deps beyond torch. Used by pretrain / sft / dpo.

Architecture in one breath: tokens -> embeddings (+ positions) -> N identical
Blocks of [causal self-attention, MLP] with pre-norm residual connections ->
final LayerNorm -> linear projection back to vocab logits. "Decoder-only" means
every token may only attend to itself and earlier tokens (causal masking), which
is exactly the autoregressive next-token-prediction objective the model is trained on.
"""

# All architecture hyperparameters live here so a model is fully described by one
# small, serializable object. Changing any of these changes the parameter shapes,
# so a checkpoint is only loadable into a model built with the same config.
@dataclass
class GPTConfig:
    vocab_size: int = 8192     # size of the token vocabulary; sets embedding rows and output logits width
    block_size: int = 256      # max context length (in tokens): the fixed window of past tokens a position can attend to
    n_layer: int = 6           # number of stacked transformer Blocks (depth); more layers = more representational capacity
    n_head: int = 6            # number of attention heads run in parallel within each attention layer
    n_embd: int = 384          # must be divisible by n_head -- the model width is split evenly across heads, so head_dim = n_embd / n_head
    dropout: float = 0.1       # regularization: randomly zeroes activations during training to prevent overfitting on small data
    bias: bool = True          # bias in Linear/LayerNorm layers (GPT-2 used biases; some later models drop them for a tiny speed/quality gain)


# Multi-head causal self-attention: the layer that lets each token mix in
# information from earlier tokens. "Causal" = it can only look backward, never forward.
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        # Width must divide evenly into heads so each head gets an equal slice (head_dim) of the representation.
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.dropout = cfg.dropout
        # one projection for q, k, v together, then output projection
        # Fusing Q, K, V into a single Linear (3*n_embd out) is a pure efficiency win:
        # one big matmul is faster than three small ones and shares a single weight tensor.
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        # Output projection: mixes the concatenated per-head outputs back into model space before the residual add.
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        # Dropout applied to the attention weights (inside sdpa) and to the residual output, both for regularization.
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape   # batch, sequence length (time), channels (= n_embd)
        # Run the fused projection, then slice the result back into the three n_embd-wide tensors.
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # (B, n_head, T, head_dim)
        # Reshape so heads become a separate axis: each head operates on its own head_dim slice of the
        # representation in parallel. This is what lets different heads specialize in different relationships
        # (e.g. one tracks syntax, another long-range topic) -- multi-head = multiple attention "subspaces".
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # flash/sdpa with causal mask; works on MPS in torch 2.x
        # sdpa computes softmax(Q @ K^T / sqrt(head_dim)) @ V. The 1/sqrt(head_dim) scaling is critical:
        # dot products of two head_dim-long vectors grow in variance with head_dim, and large logits push
        # softmax toward a one-hot distribution where gradients vanish. Dividing by sqrt(head_dim) keeps the
        # logit variance ~1 so the softmax stays soft and trainable.
        # is_causal=True applies a lower-triangular mask: position t may attend to positions 0..t only, never
        # the future. This autoregressive constraint is what lets us compute the loss at EVERY position in one
        # parallel forward pass during training, rather than feeding tokens one at a time.
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,   # attention dropout only active in train mode
            is_causal=True,
        )
        # Undo the head split: bring heads back next to head_dim and merge into a single C-wide vector per token.
        # .contiguous() is needed because transpose only changes strides, not memory layout, before .view.
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        # Project the merged heads back to model space, then apply residual dropout before this returns to the residual stream.
        return self.resid_dropout(self.c_proj(y))


# Position-wise feed-forward network. Attention moves information BETWEEN tokens;
# the MLP then processes each token's vector INDEPENDENTLY (same weights at every position),
# giving the model per-token nonlinear computation capacity.
class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        # Expand to 4x width: the standard transformer FFN ratio. The wider hidden layer gives the
        # nonlinearity room to compute richer features before projecting back down.
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        # Project the 4x hidden representation back to model width so it can be added to the residual stream.
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)

    # GELU is a smooth, differentiable-everywhere activation (a soft version of ReLU);
    # it tends to train better than ReLU in transformers. Applied identically to every position.
    def forward(self, x):
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))


# One transformer Block = attention sublayer + MLP sublayer, each wrapped in a
# pre-norm residual connection. The model is just n_layer of these stacked.
class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)   # normalizes input to the attention sublayer
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)   # normalizes input to the MLP sublayer
        self.mlp = MLP(cfg)

    # PRE-norm design: LayerNorm is applied to the input of each sublayer (norm-then-sublayer),
    # NOT to the sum afterward (post-norm, as in the original Transformer). Pre-norm leaves an
    # unobstructed "x + ..." identity path from input to output, so gradients flow cleanly through
    # many layers without exploding/vanishing -- this is what makes deep transformers trainable.
    # The residual additions themselves (x = x + sublayer(...)) are the mechanism that makes depth work:
    # each block learns a small correction to the running representation instead of replacing it.
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # pre-norm residual
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),     # token embeddings: maps each token id to a learned n_embd vector
            # LEARNED absolute position embeddings: one vector per slot 0..block_size-1. Self-attention is
            # permutation-invariant (it sees a set, not a sequence), so without adding position info the model
            # literally could not tell "dog bites man" from "man bites dog". These are added to token embeddings.
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),     # position embeddings
            drop=nn.Dropout(cfg.dropout),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),   # the stack of transformer Blocks
            ln_f=nn.LayerNorm(cfg.n_embd, bias=cfg.bias),     # final LayerNorm before projecting to logits
        ))
        # Language-model head: projects each final hidden vector to a logit per vocabulary token.
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        # weight tying: input embedding and output projection share weights
        # Tying wte and lm_head (same weight matrix) (1) saves vocab_size*n_embd parameters, and
        # (2) ties the input and output token spaces -- a token's embedding and its output classifier
        # row are the same vector -- which acts as a regularizer and usually improves quality.
        self.transformer.wte.weight = self.lm_head.weight
        # Initialize all Linear/Embedding weights with the default scheme below.
        self.apply(self._init_weights)
        # scaled init for residual projections (GPT-2 trick)
        # Each Block adds two residual contributions, so over n_layer the residual stream's variance would
        # grow ~with depth if every projection started at std 0.02. Scaling the OUTPUT projection of each
        # sublayer (c_proj) down by 1/sqrt(2*n_layer) keeps the accumulated residual variance roughly stable,
        # which stabilizes training of deep models.
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    # GPT-2's small-init scheme: a tight normal (std 0.02) for weights, zeros for biases.
    # Small initial weights keep early-training activations and gradients in a stable range.
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        # Total trainable params. parameters() dedupes by identity, so the tied
        # wte/lm_head weight is counted once, not twice.
        return sum(p.numel() for p in self.parameters())

    # Rebuild a model straight from a training checkpoint (see the torch.save calls
    # in pretrain/sft/dpo). Returns it in eval() mode, ready for generation; the
    # training scripts load by hand because they need train mode and (dpo) tweak cfg.
    @classmethod
    def from_checkpoint(cls, path, device):
        ckpt = torch.load(path, map_location=device)
        model = cls(GPTConfig(**ckpt["config"])).to(device)
        model.load_state_dict(ckpt["model"])
        return model.eval()

    def forward(self, idx, targets=None):
        B, T = idx.shape   # idx is a (batch, seq_len) tensor of integer token ids
        assert T <= self.cfg.block_size, f"seq len {T} > block_size {self.cfg.block_size}"
        # Position ids 0,1,...,T-1 to index into the learned position embedding table.
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        # Sum token + position embeddings (broadcast over batch) so each token's vector encodes both
        # WHAT it is and WHERE it sits, then apply input dropout. This sum is the start of the residual stream.
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        # Pass through every transformer Block in sequence; each refines the residual stream.
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)   # final normalization stabilizes the activations feeding the head
        logits = self.lm_head(x)       # (B, T, vocab_size): an unnormalized score per next-token candidate at each position
        loss = None
        if targets is not None:
            # Cross-entropy over the next-token prediction at every position at once (flattened to (B*T, vocab)).
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                # ignore_index=-1 drops any position whose target is -1 from the loss. Callers set target=-1
                # on tokens they don't want supervised -- e.g. SFT masks the prompt so only the response is trained.
                ignore_index=-1,   # lets SFT mask out prompt tokens
            )
        return logits, loss

    # @torch.no_grad(): inference only, so skip building the autograd graph (saves memory/time).
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_id=None):
        """Autoregressive sampling. idx is (B, T) of context tokens."""
        # Generate one token at a time, appending each sampled token back onto the context and repeating.
        for _ in range(max_new_tokens):
            # Crop to the last block_size tokens: the model has no position embeddings beyond that window.
            idx_cond = idx[:, -self.cfg.block_size:]
            # NOTE: this recomputes attention over the WHOLE sequence every step (no KV cache). That's O(T)
            # redundant work per token, traded away here for code simplicity over inference speed.
            logits, _ = self(idx_cond)
            # Keep only the last position's logits (the prediction for the next token) and apply temperature.
            # Temperature divides the logits BEFORE softmax: <1 sharpens the distribution (more greedy/confident),
            # >1 flattens it (more random/diverse). max(.,1e-6) guards against divide-by-zero at temperature 0.
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                # top-k truncation: keep only the k highest-probability tokens, set the rest to -inf so they
                # get zero probability after softmax. This prevents sampling rare, low-quality tokens from the tail.
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)   # turn logits into a normalized probability distribution
            # Sample from the distribution (stochastic) rather than taking argmax (greedy); this is what makes
            # temperature/top_k meaningful and gives varied generations.
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)   # append the new token and loop
            # Early stop once every sequence in the batch has emitted the end-of-sequence token.
            if eos_id is not None and (next_id == eos_id).all():
                break
        return idx

    def configure_optimizer(self, lr, weight_decay=0.1, betas=(0.9, 0.95)):
        # weight-decay matrices, not biases/LayerNorms/embeddings
        # Split parameters into two groups by dimensionality. 2D+ tensors are the weight matrices that do
        # the actual mixing (Linear/attention/embedding weights) and benefit from L2 regularization to keep
        # them small and generalize. 1D tensors (biases, LayerNorm gains/shifts) are per-feature offsets/scales;
        # decaying them toward zero just hurts -- it would fight the normalization and bias the model.
        decay, no_decay = [], []
        for _, p in self.named_parameters():
            if not p.requires_grad:
                continue
            # dim() >= 2 cleanly separates matrices (decay) from vectors (no decay).
            if p.dim() >= 2:
                decay.append(p)
            else:
                no_decay.append(p)
        groups = [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ]
        # AdamW = Adam with DECOUPLED weight decay: it shrinks weights directly rather than folding decay into
        # the gradient (as classic L2 + Adam would), which interacts correctly with Adam's adaptive per-parameter
        # step sizes. betas=(0.9, 0.95): the 2nd-moment beta (0.95) is lower than the usual 0.999, a common LLM
        # choice that makes the optimizer adapt faster to the noisy gradients of language-model training.
        return torch.optim.AdamW(groups, lr=lr, betas=betas)


In [16]:

# Fix the RNG so weight init, dropout, and batch sampling are reproducible.
tok = tok if tok else WordTokenizer.from_file(os.path.join(BASE_PATH, "tokenizer.json"))

cfg = GPTConfig(
    vocab_size=tok.size(),
    block_size=block_size,
    n_layer=n_layer, n_head=n_head, n_embd=n_embd,
)
model = GPT(cfg).to(device)


In [17]:
print(f"device={device}  params={model.num_params()/1e6:.1f}M  "
        f"vocab={cfg.vocab_size}  ctx={cfg.block_size}", flush=True)
print(f"model config: {cfg.__dict__}", flush=True)
print(model)


prompt = "Once upon a sad time"
n = 3 # number of samples
max_new_tokens = 160
temperature = 0.8
top_k = 50
eot = tok.token_to_id(EOT)

ids = tok.encode(prompt)
for k in range(n):
    ctx = torch.tensor([ids], device=device)
    out = model.generate(ctx, max_new_tokens=max_new_tokens,
                            temperature=temperature, top_k=top_k, eos_id=eot)
    print(out)
    print(tok.decode(out[0].tolist()))

device=cuda  params=14.6M  vocab=10000  ctx=256
model config: {'vocab_size': 10000, 'block_size': 256, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'dropout': 0.1, 'bias': True}
GPT(
  (transformer): ModuleDict(
    (wte): Embedding(10000, 384)
    (wpe): Embedding(256, 384)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=384, out_features=1152, bias=True)
          (c_proj): Linear(in_features=384, out_features=384, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=384, out_features=1536, bias=True)
          (c_proj): Linear(in_features=1536, out_features=384, bias=True)
          (dropout): Dropou

# Pretrain

## Encode Data

In [18]:
def encode_file(src, dst):
    with open(src, "r", encoding="utf-8") as f:
        text = f.read()
    docs = [d.strip() for d in text.split(EOT) if d.strip()]
    ids = []
    B = 2000  # encode in batches to keep memory flat and use fast path
    for i in range(0, len(docs), B):
        for doc_ids in tok.encode_batch(docs[i:i + B]):
            ids.extend(doc_ids)
            ids.append(tok.token_to_id(EOT))
    arr = np.array(ids, dtype=np.uint16)
    arr.tofile(dst)
    print(f"{os.path.basename(src)}: {len(docs)} stories -> "
          f"{len(arr):,} tokens -> {os.path.basename(dst)} "
          f"({arr.nbytes/1e6:.1f} MB)", flush=True)

encode_file(os.path.join(BASE_PATH, "data", "pretrain_train.txt"), os.path.join(BASE_PATH, "data", "pretrain_train.bin"))
encode_file(os.path.join(BASE_PATH, "data", "pretrain_valid.txt"), os.path.join(BASE_PATH, "data", "pretrain_val.bin"))

pretrain_train.txt: 102239 stories -> 19,853,257 tokens -> pretrain_train.bin (39.7 MB)
pretrain_valid.txt: 27630 stories -> 5,324,231 tokens -> pretrain_val.bin (10.6 MB)


## Train

In [19]:
"""Pretrain the from-scratch GPT on TinyStories (next-token prediction).

Time-budgeted: pass --minutes to cap wall-clock so it fits the hour. Prints
loss, tokens/sec, and a sample story periodically. Saves ckpt_pretrain.pt.

Theory in one paragraph: a GPT is trained as a *language model* — given a prefix
of tokens it predicts a probability distribution over the next token. The loss
is cross-entropy = the negative log-likelihood the model assigns to the true
next token, averaged over every position. Minimizing it is maximum-likelihood
estimation of the data distribution. With a ~10k-word vocabulary a model that
has learned nothing outputs a near-uniform distribution, so its loss starts near
ln(10000) ~= 9.2 nats; as it learns the statistics of English it falls toward
~2 (perplexity exp(2) ~= 7 plausible next words). That single scalar dropping is
literally the model learning the language.
"""

minutes = 10.0
max_iters = 100000
batch_size = 48

lr = 6e-4
eval_interval = 500

# Build a closure that yields random training minibatches from one .bin file.
def make_batch_fn(bin_path, block_size, batch_size, device):
    # memory-map the token stream: the file lives on disk and only the pages we
    # actually touch are paged into RAM. The pretraining corpus can be many GB
    # of uint16 token ids, far larger than memory, yet this stays cheap because
    # each step only reads batch_size*block_size of it.
    data = np.memmap(bin_path, dtype=np.uint16, mode="r")

    def get_batch():
        # Pick batch_size random start offsets into the stream. Sampling random
        # contiguous windows approximates drawing i.i.d. minibatches from the
        # data distribution, which is what SGD's convergence theory assumes. The
        # -block_size-1 guard keeps room for a full window plus its shifted target.
        ix = torch.randint(len(data) - block_size - 1, (batch_size,))
        # x = the input window [i, i+block_size).
        x = torch.stack([torch.from_numpy(data[i:i + block_size].astype(np.int64)) for i in ix])
        # y = the SAME window shifted right by one. So the target at position t is
        # the token that actually followed at t+1: this is next-token prediction.
        # Feeding the ground-truth prefix (rather than the model's own samples) is
        # "teacher forcing"; the causal mask inside the model lets all block_size
        # positions be supervised in parallel from a single forward pass.
        y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])
        # Plain synchronous host->device copy. (A real loader would pin memory and
        # pass non_blocking=True to overlap the copy with compute, but these come
        # from a numpy memmap and aren't pinned, so we don't bother here.)
        return x.to(device), y.to(device)

    return get_batch


# Learning-rate schedule: linear warmup then cosine decay to a floor.
def lr_at(it, warmup, max_iters, lr, min_lr):
    # WARMUP: ramp the LR linearly from ~0 up to the peak over the first `warmup`
    # steps. Adam's running estimates of the gradient mean/variance are based on
    # almost no data at the start, so they are noisy and biased; taking full-size
    # steps then can blow the weights up irrecoverably. A gentle ramp lets the
    # moment estimates stabilize before we trust them with large updates.
    if it < warmup:
        return lr * (it + 1) / warmup
    # After the schedule horizon, hold at the floor instead of going to zero.
    if it > max_iters:
        return min_lr
    # COSINE DECAY: anneal smoothly from peak down to min_lr. ratio goes 0 -> 1.
    ratio = (it - warmup) / max(1, (max_iters - warmup))
    # coeff: 1 at the start of decay, 0 at the end (half a cosine period). The
    # gradual, front-loaded-then-gentle taper empirically reaches lower loss and
    # generalizes better than a constant or step LR.
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    # Decay toward min_lr, not 0: a small floor keeps the model nudging into a
    # flatter minimum late in training rather than freezing the moment it lands.
    return min_lr + coeff * (lr - min_lr)


# no_grad: skip building the autograd graph during evaluation. We never call
# backward here, so storing activations for gradients would only waste memory.
@torch.no_grad()
def estimate_loss(model, batch_fns, iters=50):
    # eval() switches dropout off and makes any norm layers use running stats, so
    # the reported number reflects the deterministic inference behavior, not the
    # noisier stochastic training pass.
    model.eval()
    out = {}
    for split, fn in batch_fns.items():
        losses = torch.zeros(iters)
        # A single batch's loss is a high-variance estimate of the true loss;
        # averaging many independent batches denoises it so we can trust small
        # train-vs-val differences (and the checkpoint-on-best-val decision).
        for k in range(iters):
            x, y = fn()
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    # Restore train mode so dropout etc. resume for the next optimization steps.
    model.train()
    return out

batch_fns = {
    "train": make_batch_fn( os.path.join(BASE_PATH, "data","pretrain_train.bin"),
                            cfg.block_size, batch_size, device),
    "val": make_batch_fn(os.path.join(BASE_PATH, "data","pretrain_val.bin"),
                            cfg.block_size, batch_size, device),
}
get_batch = batch_fns["train"]
# AdamW (set up inside the model): adaptive per-parameter LRs + weight decay.
opt = model.configure_optimizer(lr)

warmup = 100
# Horizon for the cosine decay; the LR reaches its floor by this step.
sched_iters = min(max_iters, 8000)
# Floor at 1/10 of peak — a common, robust choice for the cosine minimum.
min_lr = lr / 10

t0 = time.time()
deadline = t0 + minutes * 60
tok_seen = 0
best_val = float("inf")
it = 0
last_log = t0

# Train until we hit the step cap OR run out of the wall-clock budget.
while it < max_iters and time.time() < deadline:
    # Recompute the scheduled LR for this step and push it into the optimizer.
    current_lr = lr_at(it, warmup, sched_iters, lr, min_lr)
    for g in opt.param_groups:
        g["lr"] = current_lr

    # One optimization step = forward -> loss -> backward -> clip -> update.
    x, y = get_batch()
    # Forward pass; the model returns (logits, loss) where loss is the mean
    # cross-entropy over all positions = the average negative log-likelihood.
    _, loss = model(x, y)
    # Clear stale gradients first; set_to_none=True frees the buffers (a bit
    # faster / lighter than zeroing them) so this step's grads start clean.
    opt.zero_grad(set_to_none=True)
    # Backprop: accumulate d(loss)/d(param) into each parameter's .grad.
    loss.backward()
    # Gradient clipping: rescale the whole gradient so its global L2 norm is
    # <= 1.0. A single freak batch can produce a huge gradient that takes one
    # giant step and destroys the weights; clipping caps step magnitude and
    # keeps training stable without changing the gradient's direction.
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # Apply the AdamW update using the (clipped) gradients and current LR.
    opt.step()
    # Track total tokens processed (x.numel() == batch_size * block_size).
    tok_seen += x.numel()
    it += 1

    if it % 50 == 0:
        now = time.time()
        tps = (batch_size * cfg.block_size * 50) / (now - last_log)
        last_log = now
        print(f"iter {it:5d} | loss {loss.item():.3f} | lr {current_lr:.2e} | "
                f"{tps:,.0f} tok/s | {(now-t0)/60:.1f} min", flush=True)

    # Periodic evaluation: measure denoised train/val loss. val is a genuinely
    # separate file (pretrain_val.bin), so a val loss that tracks train means
    # we're learning the language, not memorizing; val rising = overfitting.
    # (The estimate is stochastic — random windows — so best_val has some
    # noise; fine for a teaching run, but don't over-read tiny differences.)
    if it % eval_interval == 0:
        losses = estimate_loss(model, batch_fns, iters=20)
        print(f"  >> eval iter {it}: train {losses['train']:.3f}  "
                f"val {losses['val']:.3f}", flush=True)
        # Checkpoint only when val improves: keep the best generalizing model,
        # not merely the latest one (which could be mid-overfit).
        if losses["val"] < best_val:
            best_val = losses["val"]
            torch.save({"model": model.state_dict(),
                        "config": cfg.__dict__,
                        "iter": it, "val_loss": best_val},
                        "ckpt_pretrain.pt")
        # quick sample: a qualitative gut-check. As loss falls from ~9 toward
        # ~2 these samples go from gibberish to coherent little stories.
        # eval() turns dropout OFF for generation (this is inference, not a
        # training step); we switch back to train() right after.
        model.eval()
        # Seed with "Once" if it's in the vocab, else fall back to the EOT/BOS
        # token. Test `is not None` (id 0 is a real token), not truthiness.
        seed = tok.token_to_id("Once")
        if seed is None:
            seed = tok.token_to_id(EOT)
        ctx = torch.tensor([[seed]], device=device)
        out = model.generate(ctx, max_new_tokens=80, temperature=0.8, top_k=50)
        print("  sample:", tok.decode(out[0].tolist()), flush=True)
        model.train()

# final save (in case the last eval wasn't the best / never ran)
losses = estimate_loss(model, batch_fns, iters=20)
if losses["val"] < best_val:
    best_val = losses["val"]
    torch.save({"model": model.state_dict(), "config": cfg.__dict__,
                "iter": it, "val_loss": best_val},
                 "ckpt_pretrain.pt")
print(f"\nDONE  iters={it}  tok_seen={tok_seen:,}  "
        f"best_val={best_val:.3f}  time={(time.time()-t0)/60:.1f} min", flush=True)

iter    50 | loss 5.188 | lr 3.00e-04 | 35,543 tok/s | 0.3 min
iter   100 | loss 4.062 | lr 6.00e-04 | 34,631 tok/s | 0.6 min
iter   150 | loss 3.722 | lr 6.00e-04 | 33,303 tok/s | 0.9 min
iter   200 | loss 3.523 | lr 6.00e-04 | 33,137 tok/s | 1.2 min
iter   250 | loss 3.205 | lr 6.00e-04 | 32,846 tok/s | 1.5 min
iter   300 | loss 3.119 | lr 5.99e-04 | 31,823 tok/s | 1.8 min
iter   350 | loss 2.953 | lr 5.99e-04 | 30,042 tok/s | 2.2 min
iter   400 | loss 2.851 | lr 5.98e-04 | 29,969 tok/s | 2.5 min
iter   450 | loss 2.722 | lr 5.97e-04 | 30,292 tok/s | 2.9 min
iter   500 | loss 2.749 | lr 5.97e-04 | 29,968 tok/s | 3.2 min
  >> eval iter 500: train 2.657  val 2.662
  sample: Once upon a time , there was a little dog named Max . He liked to play with his friends . One day , Max was playing with his toys . He was very happy . Max asked Tim , " Hi , I ' m Max . Do you want to play with me ?" Max looked and said , " Yes , Max ! Let ' s play with the ball !" They had so much fun playing with

# Instruction Tuning

In [20]:
"""Supervised fine-tuning on TinyStories-Instruct.

Turns the pretrained base into a *promptable* model: given an instruction
block ending in "Story:", it writes a matching story. Loss is computed ONLY on
the story tokens (the prompt is masked with -1), so the model learns to
continue, not to memorize the instruction. Saves ckpt_sft.pt.

THEORY -- what SFT (supervised fine-tuning / instruction tuning) is:
The base model from pretraining is just a next-token predictor over raw text.
It "knows" language but has no notion of following an instruction -- ask it a
question and it may just continue the question. SFT keeps the *exact same*
training objective (predict the next token, cross-entropy loss) but changes the
*data*: instead of arbitrary text, we feed curated (instruction -> desired
response) pairs. By continuing next-token training on these pairs, the model's
conditional distribution shifts so that, after an instruction, the highest-
probability continuation is the kind of response we want. Nothing magic -- it is
still imitation of the corpus, but a corpus deliberately shaped like
"prompt then the answer we wish the model would give".
"""
# The boundary inside each instruct doc: everything up to and including "Story:"
# is the PROMPT (the instruction the model is conditioned on); everything after
# is the COMPLETION (the target we actually train the model to produce).


# Parse the raw instruct corpus into training examples. Each example is the
# full token sequence (prompt + completion + EOT) plus the prompt length, which
# we keep around so make_batch knows exactly which tokens to MASK from the loss.
def build_examples(path, tok, block_size, eot, limit=None):
    """Return list of (ids:list[int], prompt_len:int)."""
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    # Split the file into individual documents on the EOT separator.
    docs = [d.strip() for d in text.split(EOT) if d.strip()]
    if limit:
        docs = docs[:limit]
    prompts, comps = [], []
    for d in docs:
        # Locate the "Story:" marker -- the split point between instruction and
        # story. Docs without it are malformed for our prompt/completion setup.
        i = d.find(MARKER)
        if i == -1:
            continue
        # Prompt KEEPS the marker text ("...Story:") so at inference the model
        # sees the identical cue it was trained to respond to. Completion is
        # everything after it.
        prompts.append(d[:i + len(MARKER)])
        comps.append(d[i + len(MARKER):].strip())
    examples = []
    B = 2000
    # Encode in chunks just for tokenizer throughput; semantics are unchanged.
    for s in range(0, len(prompts), B):
        penc = tok.encode_batch(prompts[s:s + B])
        cenc = tok.encode_batch(comps[s:s + B])
        for pe, ce in zip(penc, cenc):
            # Skip empty completions -- there is nothing to supervise on.
            if len(ce) == 0:
                continue
            # The training sequence: prompt tokens, then completion tokens, then
            # the EOT. Appending EOT is what teaches the model to STOP after the
            # story rather than generating forever.
            ids = pe + ce + [eot]
            # block_size+1 tokens give block_size (input, target) pairs after the
            # shift below, so we clip to the model's context length.
            if len(ids) > block_size + 1:
                ids = ids[:block_size + 1]
            if len(ids) <= len(pe):  # truncation ate the whole completion
                continue
            # Store the sequence with its prompt length for later loss masking.
            examples.append((ids, len(pe)))
    return examples


# Assemble one padded, prompt-masked training batch from a set of example
# indices. This is where the two SFT-specific tricks live: PROMPT MASKING (so
# loss only counts on the completion) and RIGHT-PADDING ragged sequences to a
# rectangle (so they fit in one tensor).
def make_batch(examples, idxs, block_size, pad_id, device):
    # Pad only to the longest example in THIS batch (capped at context length),
    # not to block_size -- avoids wasting compute on all-pad columns.
    maxlen = min(block_size + 1, max(len(examples[i][0]) for i in idxs))
    B = len(idxs)
    # Inputs default to PAD; targets default to -1 (the ignore_index). So any
    # position we never fill -- i.e. right-padding -- is automatically excluded
    # from the loss on the target side.
    X = np.full((B, maxlen - 1), pad_id, dtype=np.int64)
    Y = np.full((B, maxlen - 1), -1, dtype=np.int64)
    for r, i in enumerate(idxs):
        ids, plen = examples[i]
        ids = ids[:maxlen]
        # The causal-LM shift: input is the sequence minus its last token, target
        # is the sequence minus its first. So x[t] predicts y[t] = ids[t+1], i.e.
        # "given tokens so far, what's next". This pairing is the whole objective.
        x = ids[:-1]
        y = ids[1:]
        X[r, :len(x)] = x
        yy = list(y)
        # PROMPT MASKING -- the key SFT idea. We zero out the loss on positions
        # that predict prompt tokens, so the model spends NO capacity learning to
        # reproduce the instruction; it only learns to generate the completion
        # CONDITIONED on that instruction. Why exactly (plen-1)? After the shift,
        # target position j predicts ids[j+1]. The prompt occupies ids[0:plen], so
        # its tokens appear as targets at j = 0 .. plen-2 (that's plen-1 of them).
        # Crucially we stop there: position plen-1 has target ids[plen], the FIRST
        # completion token -- the first real supervised signal. That prediction is
        # made from the last prompt token as context, which is exactly the
        # transition we DO want to learn, so masking stops one short.
        for j in range(min(plen - 1, len(yy))):
            yy[j] = -1
        Y[r, :len(yy)] = yy
    # Right-padding is safe under causal attention: a real token at position t can
    # only attend to positions <= t, so it never sees the pad tokens appended
    # after it; and the pad targets are -1, so they contribute nothing to loss.
    # (Left-padding would instead push real tokens to shifted positions -- fine
    # too, but it changes positional indices, so right-padding is simplest here.)
    return (torch.from_numpy(X).to(device), torch.from_numpy(Y).to(device))


@torch.no_grad()
def estimate_loss(model, examples, batch_size, block_size, pad_id, device, iters=20):
    """Mean completion-masked loss on a set of examples (e.g. the held-out split).

    eval() turns dropout OFF and no_grad saves memory; we average several batches
    to denoise. NOTE: like the training loss, this counts ONLY completion tokens
    (prompt/pad are masked to -1), so it is NOT comparable to the pretraining
    loss, which counts every token. Compare it only against itself over time:
    rising val loss while train loss falls = the model memorizing the SFT set.
    """
    model.eval()
    losses = []
    for _ in range(iters):
        idxs = rng.integers(0, len(examples), size=batch_size).tolist()
        x, y = make_batch(examples, idxs, block_size, pad_id, device)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


minutes = 12.0
max_iters = 100000
batch_size = 48
# LOWER learning rate than pretraining on purpose. SFT only needs to nudge an
# already-competent model toward instruction-following; a large LR would
# overwrite the broadly-useful weights learned in pretraining (catastrophic
# forgetting), degrading the model's general fluency. Small steps adapt the
# behavior while preserving the base language ability.
lr = 2e-4
eval_interval = 250
limit = 40000 # cap instruct examples

tok = WordTokenizer.from_file(os.path.join(BASE_PATH, "tokenizer.json"))
# Resolve the special-token ids we need: EOT to terminate completions, PAD to
# fill ragged batches. Must match the ids the model was pretrained with.
eot = tok.token_to_id(EOT)
pad_id = tok.token_to_id(PAD)

# SFT starts FROM the pretrained checkpoint -- we are adapting an existing
# model, not training from scratch. Same architecture/config, just new data.
ckpt = torch.load(os.path.join(BASE_PATH, "ckpt_pretrain.pt"), map_location=device)
cfg = GPTConfig(**ckpt["config"])
model = GPT(cfg).to(device)
model.load_state_dict(ckpt["model"])
print(f"loaded base (val_loss={ckpt['val_loss']:.3f}) on {device}", flush=True)
print(f"model config: {cfg.__dict__}", flush=True)
print(model)

examples = build_examples( os.path.join(BASE_PATH, "data","instruct_train.txt"),
                            tok, cfg.block_size, eot, limit=limit)
# Held-out split: SFT runs on a FINITE instruction set, so this is exactly
# where overfitting/memorization shows up -- val loss rising while train loss
# keeps falling is the classic signature. We also keep the best-val checkpoint.
val_examples = build_examples(os.path.join(BASE_PATH, "data","instruct_valid.txt"),
                                tok, cfg.block_size, eot, limit=5000)
print(f"SFT examples: {len(examples):,}  (held-out val {len(val_examples):,})",
        flush=True)

opt = model.configure_optimizer(lr)
t0 = time.time()
deadline = t0 + minutes * 60
it = 0
best_val = float("inf")
while it < max_iters and time.time() < deadline:
    # Sample a random batch of examples (with replacement) each step -- this
    # is plain SGD over the SFT dataset, identical in form to pretraining.
    idxs = rng.integers(0, len(examples), size=batch_size).tolist()
    x, y = make_batch(examples, idxs, cfg.block_size, pad_id, device)
    # Forward pass returns the cross-entropy loss; internally it ignores
    # target == -1, so prompt and pad positions contribute zero gradient --
    # the whole training signal comes only from the completion tokens.
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    # Gradient clipping caps the update norm, guarding against the occasional
    # large-gradient batch destabilizing the delicately-tuned base weights.
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    it += 1
    if it % 50 == 0:
        now = time.time()
        print(f"iter {it:5d} | loss {loss.item():.3f} | "
                f"{(now-t0)/60:.1f} min", flush=True)
    if it % eval_interval == 0:
        # Quantitative check: completion loss on the held-out split, and keep
        # the BEST-generalizing checkpoint rather than just the final-iter one.
        val_loss = estimate_loss(model, val_examples, batch_size,
                                    cfg.block_size, pad_id, device, iters=20)
        print(f"  >> eval iter {it}: val_loss {val_loss:.3f}", flush=True)
        if val_loss < best_val:
            best_val = val_loss
            torch.save({"model": model.state_dict(), "config": cfg.__dict__,
                        "iter": it, "val_loss": best_val},
                         "ckpt_sft.pt")
        # Qualitative check: a FIXED hand-written instruction (NOT from the val
        # split) ending in "Story:" -- the prompt format the model was trained
        # to complete. Does the generated story actually obey Words/Features?
        # eval() turns dropout off for this inference sample; train() after.
        model.eval()
        demo = ("Features: Dialogue\nWords: dog, happy, park\n"
                "Summary: A dog makes a new friend at the park.\nStory:")
        demo_ids = tok.encode(demo)
        ids = torch.tensor([demo_ids], device=device)
        # eos_id=eot lets generation stop early once the model emits the EOT
        # it learned to append -- evidence the "stop" training took effect.
        out = model.generate(ids, max_new_tokens=120, temperature=0.8,
                                top_k=50, eos_id=eot)
        # Slice off the prompt tokens so we print only the model's response.
        gen = tok.decode(out[0].tolist()[len(demo_ids):])
        print("  --- sample story ---\n  " + gen.replace("\n", "\n  "),
                flush=True)
        model.train()


print(f"\nDONE  iters={it}  best_val={best_val:.3f}  "
        f"time={(time.time()-t0)/60:.1f} min  -> ckpt_sft.pt", flush=True)


loaded base (val_loss=2.243) on cuda
model config: {'vocab_size': 10000, 'block_size': 256, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'dropout': 0.1, 'bias': True}
GPT(
  (transformer): ModuleDict(
    (wte): Embedding(10000, 384)
    (wpe): Embedding(256, 384)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=384, out_features=1152, bias=True)
          (c_proj): Linear(in_features=384, out_features=384, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=384, out_features=1536, bias=True)
          (c_proj): Linear(in_features=1536, out_features=384, bias=True)
          (dropout): Dropout(p=0.1, in

# DPO

In [21]:

minutes = 12.0
max_steps = 100000
batch_size = 16 # pairs per update
lr = 1e-5
# beta: the DPO temperature / trust-region knob. Smaller = the policy moves
# more aggressively toward the preferred completions; larger = it stays closer
# to the reference (more conservative). 0.1 is the textbook default, but with a
# tiny model and a gameable programmatic reward that over-optimizes into
# incoherence (positive/required words crammed in regardless of sense), we
# default to 0.3 to anchor harder to the SFT distribution and keep stories
# coherent. Lower it toward 0.1 if you want stronger (riskier) steering.
beta = 0.3
reward = "both" # choices=["sentiment", "words", "both"]
# pair-generation knobs
n_pairs = 2000 # preference pairs to build
k = 4 # samples per prompt (best vs worst)
gen_batch = 16 # prompts per generation call
gen_len = 80
temperature = 1.0
top_k = 50
eval_interval = 50

## Rewards

In [22]:
"""Reward functions for the DPO/RLHF stage.

Deliberately simple and transparent so the demo is legible — you can read the
reward and predict how the model will be pushed. Two rewards:

  * sentiment      : reward positive/"happy" stories, penalize sad/scary ones
  * required_words : reward including the words asked for in the "Words:" line

Both return a float per story. Combine as you like in dpo.py.
"""

POSITIVE = {
    "happy", "happily", "joy", "joyful", "smile", "smiled", "smiling", "laugh",
    "laughed", "laughing", "fun", "love", "loved", "loving", "kind", "kindly",
    "friend", "friends", "friendly", "play", "played", "playing", "good", "great",
    "wonderful", "nice", "proud", "excited", "cheer", "cheered", "hug", "hugged",
    "share", "shared", "sharing", "brave", "safe", "warm", "yay", "hooray",
    "best", "beautiful", "delighted", "glad", "grateful", "thank", "thanked",
}
NEGATIVE = {
    "sad", "sadly", "cry", "cried", "crying", "scared", "scary", "afraid",
    "angry", "mad", "hurt", "hurts", "pain", "bad", "terrible", "awful", "fear",
    "fearful", "worried", "worry", "lonely", "alone", "dark", "monster", "die",
    "died", "dead", "broken", "lost", "lose", "cruel", "mean", "fight",
    "fought", "yell", "yelled", "scream", "screamed", "frown", "frowned", "sick",
}

_WORD = re.compile(r"[a-zA-Z']+")


def _tokens(text):
    return [w.lower() for w in _WORD.findall(text)]


def sentiment_reward(text):
    """+1 per positive word, -1 per negative, normalized by length. Range ~[-1, 1]."""
    toks = _tokens(text)
    if not toks:
        return 0.0
    pos = sum(t in POSITIVE for t in toks)
    neg = sum(t in NEGATIVE for t in toks)
    # scale so a story with a healthy sprinkle of positive words approaches 1
    return max(-1.0, min(1.0, (pos - neg) / max(8.0, len(toks) / 12.0)))


def required_words_reward(prompt_text, gen_text):
    """Fraction of the prompt's "Words:" line that actually appear in the story."""
    m = re.search(r"Words:\s*(.+)", prompt_text)
    if not m:
        return 0.0
    wanted = [w.strip().lower() for w in m.group(1).split(",") if w.strip()]
    if not wanted:
        return 0.0
    gen = set(_tokens(gen_text))
    hit = sum(any(w == g or g.startswith(w) for g in gen) for w in wanted)
    return hit / len(wanted)


def length_penalty(gen_text, target=70, span=60):
    """Mild penalty for being far from a target word count (anti reward-hacking)."""
    n = len(_tokens(gen_text))
    return -min(1.0, abs(n - target) / span)


def compute_reward(prompt_text, gen_text, mode="sentiment"):
    """Combined scalar reward. mode in {sentiment, words, both}."""
    r = 0.0
    if mode in ("sentiment", "both"):
        r += sentiment_reward(gen_text)
    if mode in ("words", "both"):
        r += 0.5 * required_words_reward(prompt_text, gen_text)
    r += 0.1 * length_penalty(gen_text)
    return r


## Pairs Data

In [23]:
"""DPO (Direct Preference Optimization) on top of the SFT model.

The stable alternative to PPO. Instead of an online RL loop (sample -> score ->
policy-gradient -> hope it doesn't diverge), DPO turns preference learning into a
single *supervised* classification-style loss. There is NO value head, NO
importance ratio, NO advantage estimation, NO KL controller -- so none of the
failure modes that make PPO fragile can occur. It trains like SFT and essentially
cannot collapse.

The pipeline here:
  1. PAIR GENERATION (offline): for each prompt, sample k completions from the
     frozen SFT model, score them with rewards.py, and keep the best as "chosen"
     and the worst as "rejected". This is a programmatic stand-in for human
     preference labels -- the reward function plays the role of the human.
  2. DPO LOSS: train the policy so that, relative to the frozen reference (the SFT
     model), it raises the log-prob of `chosen` and lowers it for `rejected`:

         L = -log sigmoid( beta * [ (logπθ(y_w) - logπref(y_w))
                                    - (logπθ(y_l) - logπref(y_l)) ] )

     The bracket is the difference of "implicit rewards" DPO assigns to the two
     completions; pushing it positive = preferring chosen over rejected. beta
     controls how far the policy may move from the reference (small beta = stay
     close). Because the reference terms anchor every update, the policy can't run
     away into reward-hacked gibberish the way unconstrained PPO did.

Saves ckpt_dpo.pt. Init from ckpt_sft.pt.
"""


def load_prompts(path, tok, max_prompt_len, limit):
    """Bucket prompts by token length so each generation batch is rectangular."""
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    docs = [d.strip() for d in text.split(EOT) if d.strip()][:limit]
    buckets = {}
    for d in docs:
        i = d.find(MARKER)
        if i == -1:
            continue
        ps = d[:i + len(MARKER)]
        ids = tok.encode(ps)
        if len(ids) > max_prompt_len or len(ids) < 4:
            continue
        buckets.setdefault(len(ids), []).append((ps, ids))
    return {k: v for k, v in buckets.items() if len(v) >= 8}


def cut_at_eot(ids, eot):
    """Keep a generated completion up to and including the first EOT."""
    if eot in ids:
        return ids[:ids.index(eot) + 1]
    return ids


@torch.no_grad()
def generate_pairs(model, tok, buckets, lengths, weights, rng, minutes, max_steps, batch_size, lr, beta, reward, n_pairs, k, gen_batch, gen_len, temperature, top_k, eval_interval, eot):
    """Build a fixed buffer of (prompt_ids, chosen_ids, rejected_ids).

    For each prompt we draw k completions (sampling, so they differ), score each
    with the reward, and pair the best against the worst. Ties (all k score the
    same) carry no preference signal, so we skip them. This is pure OFFLINE DPO:
    the pairs come from the frozen SFT model once and are then reused -- the
    simplest, most stable recipe. (An "iterative DPO" variant would refresh the
    pairs from the improving policy every so often.)
    """
    model.eval()
    pairs = []
    attempts = 0
    max_attempts = n_pairs * 6
    while len(pairs) < n_pairs and attempts < max_attempts:
        L = int(rng.choice(lengths, p=weights))
        pool = buckets[L]
        pick = rng.integers(0, len(pool), size=gen_batch)
        prompt_strs = [pool[i][0] for i in pick]
        prompt_ids = [pool[i][1] for i in pick]
        attempts += gen_batch
        # repeat each prompt k times so we get k independent samples per prompt
        idx = torch.tensor(prompt_ids, device=device).repeat_interleave(k, dim=0)
        out = model.generate(idx, max_new_tokens=gen_len,
                             temperature=temperature, top_k=top_k, eos_id=None)
        comp = out[:, L:].tolist()
        for b in range(gen_batch):
            cands = []  # (reward, comp_ids) for this prompt's k samples
            for j in range(k):
                ids = cut_at_eot(comp[b * k + j], eot)
                r = compute_reward(prompt_strs[b], tok.decode(ids), mode=reward)
                cands.append((r, ids))
            cands.sort(key=lambda c: c[0])
            worst, best = cands[0], cands[-1]
            if best[0] - worst[0] > 1e-6 and best[1] and worst[1]:
                pairs.append((prompt_ids[b], best[1], worst[1]))
                if len(pairs) >= n_pairs:
                    break
    return pairs


def make_dpo_batch(pairs, idxs, pad_id, device):
    """Pack a minibatch into (chosen_seq, chosen_mask, rej_seq, rej_mask).

    Each sequence = prompt tokens + completion tokens. The mask marks the
    COMPLETION token positions (1.0); prompt and right-pad positions are 0.0 so
    only the completion contributes to the sequence log-prob. Right-padding is
    safe under causal attention (real tokens never attend to future pads).
    """
    rows = [pairs[i] for i in idxs]
    # Build the chosen and rejected sides as plain parallel lists. seq = prompt +
    # completion; mask marks the completion positions (1.0) vs prompt/pad (0.0).
    chosen_seqs, chosen_masks = [], []
    rej_seqs, rej_masks = [], []
    for prompt, chosen, rejected in rows:
        chosen_seqs.append(prompt + chosen)
        chosen_masks.append([0.0] * len(prompt) + [1.0] * len(chosen))
        rej_seqs.append(prompt + rejected)
        rej_masks.append([0.0] * len(prompt) + [1.0] * len(rejected))
    maxlen = max(len(s) for s in chosen_seqs + rej_seqs)

    # Right-pad one side's ragged sequences/masks into rectangular tensors.
    def pack(seqs, masks):
        X = np.full((len(seqs), maxlen), pad_id, dtype=np.int64)
        M = np.zeros((len(seqs), maxlen), dtype=np.float32)
        for r, (s, m) in enumerate(zip(seqs, masks)):
            X[r, :len(s)] = s
            M[r, :len(m)] = m
        return torch.from_numpy(X).to(device), torch.from_numpy(M).to(device)

    cs, cm = pack(chosen_seqs, chosen_masks)
    rs, rm = pack(rej_seqs, rej_masks)
    return cs, cm, rs, rm


def completion_logprob(model, seq, comp_mask):
    """Sum of per-token log-probs over the completion tokens of each sequence.

    logits[:, t] predict seq[:, t+1], so we score targets seq[:, 1:] with logits
    shifted by one. comp_mask[:, 1:] aligns the completion flags to those targets,
    so the result is logπ(completion | prompt) for each row -- exactly the
    quantity DPO compares between chosen and rejected, policy and reference.
    """
    logits, _ = model(seq)
    logp = F.log_softmax(logits[:, :-1, :], dim=-1)
    targets = seq[:, 1:]
    tok_logp = torch.gather(logp, 2, targets.unsqueeze(-1)).squeeze(-1)
    return (tok_logp * comp_mask[:, 1:]).sum(dim=1)


tok = tok if tok else WordTokenizer.from_file(os.path.join(BASE_PATH, "tokenizer.json"))
eot = tok.token_to_id(EOT)
pad_id = tok.token_to_id(PAD)

ckpt = torch.load(os.path.join(BASE_PATH, "ckpt_sft.pt"), map_location=device)
cfg = GPTConfig(**ckpt["config"])
cfg.dropout = 0.0   # inference-time behavior: deterministic log-probs, no mask noise

# policy = trainable, starts from SFT. reference = frozen SFT (the anchor).
policy = GPT(cfg).to(device)
policy.load_state_dict(ckpt["model"])
ref = GPT(cfg).to(device)
ref.load_state_dict(ckpt["model"])
ref.eval()
for p in ref.parameters():
    p.requires_grad_(False)

max_prompt_len = cfg.block_size - gen_len - 1
buckets = load_prompts(os.path.join(BASE_PATH, "data","instruct_valid.txt"),
                        tok, max_prompt_len, limit=20000)
lengths = list(buckets.keys())
weights = np.array([len(buckets[l]) for l in lengths], dtype=np.float64)
weights /= weights.sum()

# ---- fixed eval set + before/after reward (the demo payoff) ----
eval_pool = [pi for l in lengths for pi in buckets[l]]
eval_prompts = [eval_pool[i] for i in rng.integers(0, len(eval_pool), size=64)]
# The eval set is fixed, so group it by prompt length ONCE (rather than on every
# evaluate() call). Each length group is generated as one rectangular batch.
eval_byl = {}
for ps, ids in eval_prompts:
    eval_byl.setdefault(len(ids), []).append((ps, ids))



In [24]:

# ---- build the preference pair buffer (offline) ----
if not os.path.exists(os.path.join(BASE_PATH, "pairs.csv")):
  pairs = generate_pairs(policy, tok, buckets, lengths, weights, rng, minutes,
                        max_steps, batch_size, lr, beta, reward, n_pairs, k,
                        gen_batch, gen_len, temperature, top_k, eval_interval, eot)
  pd.DataFrame(pairs).to_csv(os.path.join(BASE_PATH, "pairs.csv"))
else:
  df = pd.read_csv(os.path.join(BASE_PATH, "pairs.csv"), index_col=0)

  # Convert string representations like "[1, 2, 3]" back to lists
  for col in df.columns:
      df[col] = df[col].apply(ast.literal_eval)

  pairs = df.values.tolist()

In [25]:
pd.DataFrame(pairs)

,0,1,2
0,"[83, 21, 158, 987, 1981, 7, 428, 108, 909, 116...","[60, 61, 7, 47, 4, 49, 9, 7, 987, 132, 32, 7, ...","[60, 61, 7, 47, 4, 49, 9, 7, 987, 3, 18, 9, 50..."
1,"[83, 21, 7365, 56, 2869, 5, 443, 710, 7, 274, ...","[60, 61, 7, 47, 4, 49, 9, 7, 37, 88, 55, 2744,...","[60, 61, 7, 47, 4, 49, 9, 7, 56, 132, 180, 20,..."
2,"[126, 21, 1086, 4, 1884, 4, 240, 83, 21, 8657,...","[60, 61, 7, 47, 4, 49, 9, 7, 37, 85, 55, 95, 3...","[60, 61, 7, 47, 4, 7, 37, 85, 55, 1334, 62, 39..."
3,"[83, 21, 25, 3020, 7, 212, 459, 39, 14, 201, 4...","[60, 61, 7, 47, 4, 20, 7, 1590, 37, 416, 4, 49...","[45, 540, 23, 4, 25, 9, 20, 6, 409, 3, 18, 429..."
4,"[83, 21, 158, 37, 68, 1756, 237, 662, 20, 7, 8...","[60, 61, 7, 47, 4, 49, 9, 7, 37, 68, 55, 472, ...","[60, 49, 9, 7, 37, 68, 132, 32, 7, 662, 30, 24..."
...,...,...,...
1995,"[148, 21, 189, 4, 698, 230, 229, 21, 45, 23, 4...","[60, 61, 7, 47, 4, 49, 9, 7, 37, 66, 55, 25, 3...","[60, 61, 7, 47, 4, 49, 9, 7, 274, 66, 55, 25, ..."
1996,"[126, 21, 436, 4, 1706, 4, 283, 83, 21, 574, 4...","[60, 61, 7, 47, 4, 49, 9, 7, 37, 66, 55, 574, ...","[60, 61, 7, 47, 4, 49, 9, 7, 37, 372, 327, 574..."
1997,"[126, 21, 1082, 4, 2183, 4, 1032, 83, 21, 1659...","[60, 61, 7, 47, 4, 49, 9, 7, 27, 4, 1032, 359,...","[60, 61, 7, 47, 4, 49, 9, 144, 1032, 359, 3, 1..."
1998,"[83, 21, 53, 5, 194, 87, 33, 7, 2850, 17, 67, ...","[53, 5, 194, 75, 1885, 3, 12, 101, 8, 43, 20, ...","[45, 23, 4, 7, 37, 88, 55, 53, 5, 194, 62, 8, ..."


## DPO Training

In [26]:
@torch.no_grad()
def evaluate(show=2):
    policy.eval()
    rewards = []
    for L, items in eval_byl.items():
        idx = torch.tensor([ids for _, ids in items], device=device)
        out = policy.generate(idx, max_new_tokens=gen_len,
                                temperature=0.8, top_k=top_k, eos_id=None)
        for r, (ps, _) in enumerate(items):
            g = cut_at_eot(out[r].tolist()[L:], eot)
            rewards.append(compute_reward(ps, tok.decode(g), mode=reward))
    policy.train()
    mean_r = float(np.mean(rewards)) if rewards else 0.0
    if show:
        ps, ids = eval_prompts[0]
        out = policy.generate(torch.tensor([ids], device=device),
                                max_new_tokens=gen_len, temperature=0.8,
                                top_k=top_k, eos_id=eot)
        g = cut_at_eot(out[0].tolist()[len(ids):], eot)
        print("  sample:", tok.decode(g)[:300], flush=True)
    return mean_r

t0 = time.time()

policy.train()
print(f"built {len(pairs)} preference pairs in {(time.time()-t0)/60:.1f} min", flush=True)

opt = torch.optim.AdamW(policy.parameters(), lr=lr)
deadline = t0 + minutes * 60
best_r = 0
step = 0
while step < max_steps and time.time() < deadline:
    idxs = rng.integers(0, len(pairs), size=batch_size).tolist()
    cs, cm, rs, rm = make_dpo_batch(pairs, idxs, pad_id, device)

    # policy log-probs (trainable) and reference log-probs (frozen).
    pi_c = completion_logprob(policy, cs, cm)
    pi_r = completion_logprob(policy, rs, rm)
    with torch.no_grad():
        ref_c = completion_logprob(ref, cs, cm)
        ref_r = completion_logprob(ref, rs, rm)

    # DPO loss: prefer chosen over rejected by the reference-anchored margin.
    logits = beta * ((pi_c - ref_c) - (pi_r - ref_r))
    loss = -F.logsigmoid(logits).mean()
    # accuracy = fraction of pairs the policy already ranks correctly; the
    # implicit-reward margin shows how strongly it separates them.
    acc = (logits > 0).float().mean()

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step()
    step += 1

    if step % 10 == 0:
        print(f"step {step:4d} | loss {loss.item():.4f} | pref_acc {acc.item():.2f} "
                f"| margin {logits.mean().item():+.3f} | {(time.time()-t0)/60:.1f} min",
                flush=True)
    if step % eval_interval == 0:
        r = evaluate(show=2)
        print(f"[eval] step {step}  mean_reward={r:+.3f}", flush=True)
        if r > best_r:
            best_r = r
            torch.save({"model": policy.state_dict(), "config": cfg.__dict__,
                        "reward_mode": reward, "step": step},
                        "ckpt_dpo.pt")

final_r = evaluate(show=3)
if final_r > best_r or not os.path.exists("ckpt_dpo.pt"):
    best_r = max(best_r, final_r)
    torch.save({"model": policy.state_dict(), "config": cfg.__dict__,
                "reward_mode": reward, "step": step},
                "ckpt_dpo.pt")

print(f"\nDONE  steps={step} "
        f"best_reward={best_r:+.3f}  -> ckpt_dpo.pt", flush=True)


built 2000 preference pairs in 0.0 min
step   10 | loss 0.5764 | pref_acc 0.75 | margin +0.377 | 0.0 min
step   20 | loss 0.3759 | pref_acc 0.94 | margin +0.919 | 0.1 min
step   30 | loss 0.3785 | pref_acc 0.88 | margin +1.103 | 0.1 min
step   40 | loss 0.3330 | pref_acc 0.88 | margin +1.563 | 0.1 min
step   50 | loss 0.1776 | pref_acc 1.00 | margin +2.623 | 0.2 min
  sample: Once upon a time , there was a pretty planet . The planet was very pretty and loved to play with all day . The planet had many friends to play with . They would play together every day . The rocket was very happy because it made the little boy happy . One day , the rocket saw a new animal playing wi
[eval] step 50  mean_reward=+0.778
step   60 | loss 0.1695 | pref_acc 1.00 | margin +2.022 | 0.4 min
step   70 | loss 0.1372 | pref_acc 1.00 | margin +2.292 | 0.5 min
step   80 | loss 0.2486 | pref_acc 0.88 | margin +2.599 | 0.5 min
step   90 | loss 0.1568 | pref_acc 0.94 | margin +2.708 | 0.5 min
step  100 | loss 0.23

# Useful

In [27]:
ckpts = ["ckpt_pretrain.pt", "ckpt_sft.pt", "ckpt_dpo.pt"]
prompt = "Words: dragon. Once upon a sad time"
n = 3 # number of samples
max_new_tokens = 160
temperature = 0.8
top_k = 50
tok = WordTokenizer.from_file(os.path.join(BASE_PATH,"tokenizer.json"))
eot = tok.token_to_id(EOT)

for ckpt in ckpts:
  print(f"\n {ckpt} \n")
  model = GPT.from_checkpoint(os.path.join(BASE_PATH, ckpt), device)

  start = prompt
  ids = tok.encode(start)
  plen = len(ids)
  for k in range(n):
      ctx = torch.tensor([ids], device=device)
      out = model.generate(ctx, max_new_tokens=max_new_tokens,
                              temperature=temperature, top_k=top_k, eos_id=eot)
      full = out[0].tolist()
      if prompt is not None:
          body = tok.decode(full[plen:])
          print(f"\n===== sample {k+1} =====")
          print(start + "\n" + body)
      else:
          print(f"\n===== sample {k+1} =====")
          print(tok.decode(full))



 ckpt_pretrain.pt 


===== sample 1 =====
Words: dragon. Once upon a sad time
, there was a big bear named Bob . Bob lived in a place with a lot of food . He had a big , yummy meal for Bob to eat and eat food . Bob was very happy that he had a tasty treat . Bob ' s mom told him , " Sally , you must always tell the truth and to tell you to share the food with others ." Bob felt good , but he enjoyed eating the food . From that day on , Bob and Sally became best friends and always shared their food and treats together .

===== sample 2 =====
Words: dragon. Once upon a sad time
, there was a little girl named Lily . She had lost her toy . Lily was sad because she lost her toy . She wanted to find it . She asked her mom , " Why are you sad ?" Her mom gave her the toy . Lily knew that Max needed to find his toy to find it . She went back to her toy and found it on the floor . She was very happy and proud of her toy . Lily learned that it is important to be careful with things .

===== samp